# Automated Data-Quality Checks

**Project:** Automated Data-Quality framework for the **HEXA VPN** app &nbsp;&middot;&nbsp; **Built at:** Hazel Mobile &nbsp;&middot;&nbsp; **Data shown:** synthetic / illustrative

<div style="background:#eff6ff;border:1px solid #bfdbfe;color:#1e40af;padding:10px 14px;border-radius:8px;font-family:sans-serif;font-size:13px"><b>Data-quality framework I built at Hazel Mobile for the HEXA VPN app.</b> Figures here are <b>synthetic / illustrative</b> for portfolio use &mdash; it shows the <i>method</i>, <b>not real client, user, or production data</b>.</div>

**What this is:** a plain-language guide to an automated data-quality (DQ) framework I designed at Hazel Mobile for the **HEXA VPN** app's **connection/session telemetry**. It explains *what each check looks for* and *why it matters*, for a non-technical reader. Every check is **read-only** &mdash; it never changes the source data.

**Why it matters:** a headline "success rate" often counts every session that *reported* success. Some of those aren't real wins &mdash; a session that dropped in seconds, a corrupt record, a bot, or a user who never came back. These checks find those cases and **reclassify** them, so the business sees the *true* health of the metric, not the flattering version.

**The headline it produces (illustrative):** a raw **~92%** success rate can fall to a true **~68%** once the checks are applied &mdash; a ~24-point gap that this framework measures and explains.

## How to read this

- **Three tiers of checks**, from "is the data clean?" up to "did the user actually have a good experience?":
  1. **Row-level (#1&ndash;#11)** &mdash; is this single record clean and usable?
  2. **Behavioural (B1&ndash;B10)** &mdash; is this *account* behaving like a normal human user?
  3. **Journey (T1&ndash;T3)** &mdash; did the user actually get (and keep) a good session?
- **"Flag" / "Fail"** = the check caught something. Row-level = one bad record; behavioural/journey = a whole user or a fleet-wide alert.
- **"User"** = one account (identified by the account id inside the session token, so multiple devices/sessions roll up together).
- **"NOW"** = the most recent event in the data (not wall-clock), so windows like "last 7 days" are measured against the data itself.
- **"Fleet alert"** = a whole-product warning (e.g. "activation dropped below target"), not a per-record failure.

## Tier 1 &mdash; Row-level checks (is this one record clean?)

Each looks at a **single record** and asks "is this a trustworthy row?"

<table style="border-collapse:collapse;font-family:sans-serif;font-size:12px;border:1px solid #cbd5e1"><tr style="background:#0f172a;color:#e2e8f0"><th style="padding:6px 9px;text-align:center">#</th><th style="padding:6px 9px;text-align:left">Check</th><th style="padding:6px 9px;text-align:left">In plain words</th><th style="padding:6px 9px;text-align:left">It flags a record when&hellip;</th></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#1</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Timestamp validity</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">The clock is wrong &mdash; the event was "received" before it "happened".</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">received time is before the event time (or a timestamp is missing).</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#2</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Valid country code</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">The country isn't a real 2-letter code.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">country is present but not a valid ISO code.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#3</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Country present</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">The country is blank.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">country is missing entirely.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#4</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Non-negative duration</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Session length is negative &mdash; impossible, so corrupt.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">duration is below 0.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#5</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Non-zero duration</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Session length is exactly zero &mdash; it never really connected.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">duration equals 0.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#6</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Handshake completed</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Too short to be a real connection (under a tenth of a second).</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">duration is under 100 ms.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#7</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Valid server IP</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A "success" has no real server address &mdash; a logging problem.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">the row is a success but the server address is missing/invalid.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#8</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Collection continuity</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">The data feed went silent &mdash; a possible outage.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">there's a gap of more than 60 minutes before this record.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#9</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Plausible session length</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Too short for the user to have actually used it.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">under 14s for ad-supported users, under 4s for others.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#10</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Duration present</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">No recorded length, so it can't be judged.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">the duration field is missing.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">#11</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Self-replace bug (new)</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A "success" that was instantly replaced and the same user reconnected within a minute &mdash; a known app bug, not a real session.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">the session ended with a "replaced" reason and the same user's next event is within 60 seconds.</td></tr></table>

> **Note:** "switching servers" is **not** a failure on its own &mdash; a single failover is normal. It only counts in **B10** (switched *and then* churned).

## Tier 2 &mdash; Behavioural checks (is this account normal?)

These look at a **whole account** and ask "is this a genuine, healthy user?"

<table style="border-collapse:collapse;font-family:sans-serif;font-size:12px;border:1px solid #cbd5e1"><tr style="background:#0f172a;color:#e2e8f0"><th style="padding:6px 9px;text-align:center">ID</th><th style="padding:6px 9px;text-align:left">Check</th><th style="padding:6px 9px;text-align:left">In plain words</th><th style="padding:6px 9px;text-align:left">It flags / alerts when&hellip;</th></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B1</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Return / repeat</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">How many users come back for a second session.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Fleet alert: repeat rate below target.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B2</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">One-session share</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Share of users who connected once and never returned.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Fleet alert: one-session share above threshold.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B3</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Tenure success-floor</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">New users whose success rate is too low for how many times they tried.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">1st try &lt; 70%, 2nd &lt; 50%, or 3rd &lt; 40% success.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B4</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Very-churny segments</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Segments where almost everyone is one-and-done.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">any segment where the one-session share is 60%+.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B5</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Inactivity churn (7d)</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A user has gone quiet &mdash; no activity for a week.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">no session in the last 7 days.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B6</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Bot / burst deep-dive (new)</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Traffic that doesn't look human: an automated burst, or a user stuck reconnecting.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0"><b>Likely-bot:</b> many sessions/minute across many servers, zero failures. <b>Or reconnect-churn:</b> repeated failures on just 1&ndash;2 servers.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B7</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Rapid location hopping</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">One account appearing in many locations almost at once &mdash; impossible for one device.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">more than 4 locations within 1 minute.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B8</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Plan validity</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">The account's plan (free/premium) is missing or invalid.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">plan field is null or out of range.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B9</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Failure with no reason</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A failure was logged but no reason recorded &mdash; a logging gap.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">outcome is failure but the reason is blank.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">B10</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Churn-after-switch</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A user who had to switch servers and then never came back.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">switched at least once and then inactive for 7 days.</td></tr></table>

## Tier 3 &mdash; Journey checks (did the user have a good experience?)

These judge the **real** experience &mdash; including the "fake successes" a plain success flag hides.

<table style="border-collapse:collapse;font-family:sans-serif;font-size:12px;border:1px solid #cbd5e1"><tr style="background:#0f172a;color:#e2e8f0"><th style="padding:6px 9px;text-align:center">ID</th><th style="padding:6px 9px;text-align:left">Check</th><th style="padding:6px 9px;text-align:left">In plain words</th><th style="padding:6px 9px;text-align:left">It flags / alerts when&hellip;</th></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">T1</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Flap</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A "success" that dropped almost immediately &mdash; looked connected but didn't hold.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">the session held for under 30 seconds. Fleet alert if flaps exceed 10% of successes.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">T2a</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Activation</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Did the user's very first session actually work?</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Fleet alert: activation below target.</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">T2b</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">D1 next-day return</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Did the user come back the next day?</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Fleet alert: next-day return below target (needs 2+ days of data).</td></tr><tr><td style="padding:6px 9px;text-align:center;border:1px solid #e2e8f0">T3</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Churn-risk</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Users trying a lot but mostly failing &mdash; about to quit.</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">5+ attempts and under 30% success.</td></tr></table>

## What the framework produces (the "statements")

Running the full check suite outputs five sections. In plain terms:

<table style="border-collapse:collapse;font-family:sans-serif;font-size:12px;border:1px solid #cbd5e1"><tr style="background:#0f172a;color:#e2e8f0"><th style="padding:6px 9px;text-align:left">Section</th><th style="padding:6px 9px;text-align:left">What it gives you</th></tr><tr><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">0 &mdash; Impact (before vs after)</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">The headline: events, users, successes, failures, and <b>success % before vs after</b> the checks. Splits the users we "lose" into <i>one-and-done</i> vs <i>retained</i>.</td></tr><tr><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">1 &mdash; Checks summary</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">One row <b>per check</b>: how many records/users it flagged, the rule in words, and a status (OK / FLAGGED / ALERT). A scoreboard.</td></tr><tr><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">2 &mdash; Self-check</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">A built-in audit: re-calculates key numbers a <b>second, independent way</b> and reports <b>PASS/FAIL</b>, so the results can be trusted.</td></tr><tr><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">3 &mdash; Per-check contribution</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Ranks <b>which check caused how much</b> of the drop. Each flipped success is owned by one check, so the pieces add up.</td></tr><tr><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">4 &mdash; Record dumps</td><td style="padding:6px 9px;text-align:left;border:1px solid #e2e8f0">Pulls the <b>actual offending records</b> for each flagged check, so anyone can review or download the specific rows.</td></tr></table>

**Trust note:** every number is produced by SQL (not estimated), and the self-check re-verifies the maths, so the figures are exact and reproducible.

## Bottom line (for a quick read)

- A reported **~92%** success rate is the *face-value* number. It counts flaps, instant self-replaces, bots, bad records, and never-returning users as "success".
- Applying these checks reclassifies those cases and gives a **true** success rate of **~68%** &mdash; a ~24-point gap.
- **One check dominates:** *inactivity churn (B5)* &mdash; users who connected but never came back &mdash; accounts for the large majority of the drop. The row-level integrity checks add most of the rest; the long tail catches smaller but genuine data problems.
- Everything is **read-only** and **self-validated**, so it's safe to run against a live database and the numbers can be trusted.

*Method demonstration on illustrative data &mdash; no real client, product, or production figures are shown.*